# Single FOAR 3D 检查 JSON -> Robot Renderer

这个 notebook 参照 `ca-pi-xie-device-tools/inspect_json_robot_render_inline_no_side_predefined.ipynb`，但改成了 **单臂 FOAR / zihao purple box** 数据。

用途：
- 读取单臂 `result.json`；
- 从 `lowdim.h5` 读取指定时间戳附近的单臂 joint / tcp；
- 从真实 depth 重建点云；
- 把单臂 URDF mesh 和点云放到同一个 3D Plotly 视图里；
- 对比 `robot` / `robot_old` 两套 URDF；
- 可选追加一个手动 `ZYX` base 安装角，排查 base 定义是否还差一层。

默认先用 scene_0001 的中间帧：`1774521917807`。


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import h5py
import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from PIL import Image
from scipy.spatial.transform import Rotation as R

WORKSPACE_ROOT = Path('/home/haoxiang/rise2_mask_aware')
AIREXO_ROOT = WORKSPACE_ROOT / 'airexo'
for p in [WORKSPACE_ROOT, AIREXO_ROOT]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

from airexo.helpers.constants import (
    O3D_RENDER_TRANSFORMATION,
    ROBOT_PREDEFINED_TRANSFORMATION,
    ROBOT_TCP_TO_FLANGE,
)
from airexo.helpers import urdf_robot as robot_helper


In [ ]:
# ===== 用户参数 =====
RESULT_JSON = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/calib/result.json')
H5_PATH = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/train/scene_0001/lowdim/lowdim.h5')
SCENE_CAM_DIR = Path('/data/haoxiang/data/zihao_foar2/flip_0326_purple_box/train/scene_0001/cam_104422070117')

CAMERA_TIMESTAMP = 1774521917807
ARM_SUFFIX = '062770'
EXACT_H5_TIMESTAMP = False

JOINT_SOURCE_MODE = 'h5'   # 'h5' | 'manual_deg'
MANUAL_JOINT_DEG = [0, 0, 0, 0, 0, 0, 0]
MANUAL_GRIPPER_WIDTH = 0.0

# 与 ca-pi-xie 一样，json 语义固定，先按这个模式看。
CAM_BASE_MODE = 'base_to_cam__predef'   # 'base_to_cam__predef' | 'base_to_cam__raw' | 'cam_to_base__predef' | 'cam_to_base__raw'

URDF_CANDIDATES = {
    'robot': str((WORKSPACE_ROOT / 'airexo/airexo/urdf_models/robot/left_robot.urdf').resolve()),
    'robot_old': str((WORKSPACE_ROOT / 'airexo/airexo/urdf_models/robot_old/left_robot.urdf').resolve()),
    'zihao_single_worldbase_y_neg90': str((WORKSPACE_ROOT / 'airexo/airexo/urdf_models/zihao_single_worldbase_y_neg90/left_robot.urdf').resolve()),
}
ACTIVE_URDF_KEY = 'zihao_single_worldbase_y_neg90'
# ACTIVE_URDF_KEY = 'zihao_single_downward_baseframe_x_from_neg_z'
BASE_AXIS_REMAP_MODE = 'raw'
COMPARE_BASE_AXIS_REMAP_MODES = ['raw']
# ACTIVE_URDF_KEY = 'zihao_single_worldbase_y_pos90'

ACTIVE_URDF_KEY = 'zihao_single_downward_baseaxis_z_pos90'
# ACTIVE_URDF_KEY = 'zihao_single_downward_baseaxis_x_neg90'
# ACTIVE_URDF_KEY = 'zihao_single_downward_baseaxis_y_pos90'
# ACTIVE_URDF_KEY = 'zihao_single_downward_baseaxis_y_neg90'
# ACTIVE_URDF_KEY = 'zihao_single_worldbase_y_pos90'
# ACTIVE_URDF_KEY = 'zihao_single_downward_baseaxis_z_neg90'

# 如果怀疑还差一层 base 安装角，可以打开这个选项。
# 注意：顺序是 ZYX，输入格式写成 [Rz, Ry, Rx] 更直观。
APPLY_EXTRA_BASE_RPY_ZYX_DEG = False
EXTRA_BASE_RPY_ZYX_DEG = [90.0, 40.0, 33.0]

SHOW_GRIPPER_LINKS = False
SHOW_FRAME_AXES = True
SHOW_TCP_FRAME = False
FRAME_AXIS_LEN = 0.08

DEPTH_SCALE = 1000.0
MIN_DEPTH_M = 0.05
MAX_DEPTH_M = 1.50
POINT_STRIDE = 4
POINT_MAX = 100000
MESH_FACE_LIMIT = 20000
APPLY_O3D_TO_POINT_CLOUD = True


In [ ]:
class JointCfg:
    def __init__(self, num_joints=8, num_robot_joints=7):
        self.num_joints = num_joints
        self.num_robot_joints = num_robot_joints

JOINT_CFGS = JointCfg()
GRIPPER_KEYWORDS = ('finger', 'knuckle', 'robotiq', 'flange')


def invert_T(T):
    T = np.asarray(T, dtype=np.float64)
    out = np.eye(4, dtype=np.float64)
    out[:3, :3] = T[:3, :3].T
    out[:3, 3] = -T[:3, :3].T @ T[:3, 3]
    return out


def pose7_wxyz_to_mat(pose7):
    pose7 = np.asarray(pose7, dtype=np.float64).reshape(7)
    mat = np.eye(4, dtype=np.float64)
    qw, qx, qy, qz = pose7[3:]
    mat[:3, :3] = R.from_quat([qx, qy, qz, qw]).as_matrix()
    mat[:3, 3] = pose7[:3]
    return mat


def load_json_pose_and_intrinsic(path: Path):
    data = json.loads(path.read_text())
    T_base_to_cam = pose7_wxyz_to_mat(data['pose_in_link'])
    intrinsic = np.asarray(data['intrinsics'], dtype=np.float64)
    return T_base_to_cam, intrinsic, data


def cam_base_from_json(T_json, mode):
    robot_predef_inv = invert_T(np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64))
    if mode == 'base_to_cam__predef':
        return invert_T(T_json) @ robot_predef_inv
    if mode == 'base_to_cam__raw':
        return invert_T(T_json)
    if mode == 'cam_to_base__predef':
        return T_json @ robot_predef_inv
    if mode == 'cam_to_base__raw':
        return T_json
    raise ValueError(f'invalid cam_base_mode: {mode}')


def extra_base_from_zyx_deg(rpy_zyx_deg):
    rz, ry, rx = [float(v) for v in rpy_zyx_deg]
    mat = np.eye(4, dtype=np.float64)
    mat[:3, :3] = R.from_euler('zyx', [rz, ry, rx], degrees=True).as_matrix()
    return mat


def find_h5_index(h5_timestamps: np.ndarray, camera_timestamp: int, exact: bool = False):
    if exact:
        hit = np.where(h5_timestamps == int(camera_timestamp))[0]
        if len(hit) == 0:
            raise ValueError(f'exact timestamp {camera_timestamp} not found in h5')
        return int(hit[0])

    insert_idx = int(np.searchsorted(h5_timestamps, int(camera_timestamp)))
    if insert_idx <= 0:
        return 0
    if insert_idx >= len(h5_timestamps):
        return len(h5_timestamps) - 1
    left_idx = insert_idx - 1
    right_idx = insert_idx
    left_diff = abs(int(h5_timestamps[left_idx]) - int(camera_timestamp))
    right_diff = abs(int(h5_timestamps[right_idx]) - int(camera_timestamp))
    return left_idx if left_diff <= right_diff else right_idx


def extract_joint_and_tcp(h5_path: Path, camera_timestamp: int, arm_suffix: str, exact_ts: bool):
    with h5py.File(h5_path, 'r') as f:
        h5_timestamps = np.asarray(f['timestamp'][:], dtype=np.int64)
        idx = find_h5_index(h5_timestamps, camera_timestamp, exact=exact_ts)
        matched_ts = int(h5_timestamps[idx])
        joint7 = np.asarray(f[f'joint_position_rad_{arm_suffix}'][idx], dtype=np.float64)
        gripper = float(np.asarray(f[f'ee_state_{arm_suffix}'][idx]).reshape(-1)[0])
        tcp = np.asarray(f[f'tcp_pose_{arm_suffix}'][idx], dtype=np.float64)
    joint = np.concatenate([joint7, [gripper]], axis=0)
    return idx, matched_ts, joint, tcp


def manual_joint_from_deg(joint_deg, gripper_width):
    joint_deg = np.asarray(joint_deg, dtype=np.float64).reshape(7)
    return np.concatenate([np.deg2rad(joint_deg), [float(gripper_width)]], axis=0)


def load_scene_images(scene_dir: Path, camera_timestamp: int):
    color_path = scene_dir / 'color' / f'{camera_timestamp}.png'
    depth_path = scene_dir / 'depth' / f'{camera_timestamp}.png'
    color = np.array(Image.open(color_path).convert('RGB'))
    depth = np.array(Image.open(depth_path))
    return color_path, depth_path, color, depth


def depth_to_point_cloud(depth_mm: np.ndarray, intrinsic: np.ndarray, stride: int, depth_scale: float, min_depth_m: float, max_depth_m: float, point_max: int, apply_o3d: bool):
    fx = intrinsic[0, 0]
    fy = intrinsic[1, 1]
    cx = intrinsic[0, 2]
    cy = intrinsic[1, 2]

    depth = depth_mm.astype(np.float64) / depth_scale
    depth = depth[::stride, ::stride]
    h, w = depth.shape
    ys, xs = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    xs = xs * stride
    ys = ys * stride

    valid = np.isfinite(depth) & (depth > min_depth_m) & (depth < max_depth_m)
    z = depth[valid]
    x = (xs[valid] - cx) * z / fx
    y = (ys[valid] - cy) * z / fy
    pts = np.stack([x, y, z], axis=1)

    if apply_o3d:
        pts_h = np.concatenate([pts, np.ones((pts.shape[0], 1), dtype=np.float64)], axis=1)
        pts = (np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64) @ pts_h.T).T[:, :3]

    if pts.shape[0] > point_max:
        idx = np.linspace(0, pts.shape[0] - 1, point_max).astype(np.int64)
        pts = pts[idx]
    return pts


def should_keep_link(link_name: str):
    if SHOW_GRIPPER_LINKS:
        return True
    return not any(keyword in str(link_name).lower() for keyword in GRIPPER_KEYWORDS)


def resolve_mesh_path(mesh_rel_path: str, urdf_file: str):
    urdf_parent = Path(urdf_file).parent
    candidates = [
        urdf_parent / mesh_rel_path,
        WORKSPACE_ROOT / 'airexo/airexo/urdf_models/robot' / mesh_rel_path,
        WORKSPACE_ROOT / 'airexo/airexo/urdf_models/robot_old' / mesh_rel_path,
    ]
    for cand in candidates:
        if cand.exists():
            return cand
    raise FileNotFoundError(f'cannot resolve mesh path: {mesh_rel_path} from urdf={urdf_file}')


def load_mesh_vertices_faces(mesh_rel_path: str, urdf_file: str):
    import trimesh
    mesh_path = resolve_mesh_path(mesh_rel_path, urdf_file)
    mesh = trimesh.load_mesh(mesh_path, process=False)
    if hasattr(mesh, 'geometry'):
        mesh = trimesh.util.concatenate(tuple(mesh.geometry.values()))
    vertices = np.asarray(mesh.vertices, dtype=np.float64)
    faces = np.asarray(mesh.faces, dtype=np.int32)
    return vertices, faces


def apply_transform(vertices: np.ndarray, T: np.ndarray):
    homo = np.concatenate([vertices, np.ones((vertices.shape[0], 1), dtype=np.float64)], axis=1)
    out = (T @ homo.T).T
    return out[:, :3]


def downsample_faces(faces: np.ndarray, limit: int):
    if faces.shape[0] <= limit:
        return faces
    idx = np.linspace(0, faces.shape[0] - 1, limit).astype(np.int64)
    return faces[idx]


def add_frame(fig, T, name, axis_len=0.08):
    origin = T[:3, 3]
    axes = T[:3, :3]
    colors = ['red', 'green', 'blue']
    labels = ['x', 'y', 'z']
    for i in range(3):
        p1 = origin
        p2 = origin + axes[:, i] * axis_len
        fig.add_trace(go.Scatter3d(
            x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
            mode='lines',
            line=dict(color=colors[i], width=6),
            name=f'{name}_{labels[i]}',
            showlegend=False,
        ))


def safe_link_tf(tf_map, key):
    if key not in tf_map:
        return None
    return np.asarray(tf_map[key].matrix(), dtype=np.float64)


def fk_tcp_candidates(joint, joint_cfgs, urdf_file):
    tf_map = robot_helper.forward_kinematic_single(
        joint=joint.astype(np.float32),
        joint_cfgs=joint_cfgs,
        is_rad=True,
        urdf_file=urdf_file,
        with_visuals_map=False,
    )
    cand = {}
    flange = safe_link_tf(tf_map, 'flange')
    link7 = safe_link_tf(tf_map, 'link7')
    if flange is not None:
        cand['flange'] = flange
        cand['tcp_from_flange'] = flange @ invert_T(np.asarray(ROBOT_TCP_TO_FLANGE, dtype=np.float64))
    if link7 is not None:
        cand['link7'] = link7
    return cand


def compose_mesh_tf(cam_to_base, transform, offset, extra_base_T=None):
    base_tf = (
        np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
        @ np.asarray(cam_to_base, dtype=np.float64)
        @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
    )
    if extra_base_T is not None:
        base_tf = base_tf @ np.asarray(extra_base_T, dtype=np.float64)
    return base_tf @ transform.matrix() @ offset.matrix()


def compose_fk_world(cam_to_base, fk_T, extra_base_T=None):
    base_tf = (
        np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64)
        @ np.asarray(cam_to_base, dtype=np.float64)
        @ np.asarray(ROBOT_PREDEFINED_TRANSFORMATION, dtype=np.float64)
    )
    if extra_base_T is not None:
        base_tf = base_tf @ np.asarray(extra_base_T, dtype=np.float64)
    return base_tf @ np.asarray(fk_T, dtype=np.float64)


def compose_real_base_world(T_base_to_cam):
    return np.asarray(O3D_RENDER_TRANSFORMATION, dtype=np.float64) @ invert_T(T_base_to_cam)


def remap_base_axes(T, mode):
    T = np.asarray(T, dtype=np.float64)
    remaps = {
        'raw': np.eye(4, dtype=np.float64),
    }
    if mode not in remaps:
        raise ValueError(f'invalid base axis remap mode: {mode}')
    return T @ remaps[mode]


def compose_real_tcp_world(T_base_to_cam, tcp_pose7):
    return compose_real_base_world(T_base_to_cam) @ pose7_wxyz_to_mat(tcp_pose7)


In [ ]:
T_base_to_cam, intrinsic, raw_json = load_json_pose_and_intrinsic(RESULT_JSON)
cam_to_base = cam_base_from_json(T_base_to_cam, CAM_BASE_MODE)
extra_base_T = extra_base_from_zyx_deg(EXTRA_BASE_RPY_ZYX_DEG) if APPLY_EXTRA_BASE_RPY_ZYX_DEG else None

idx, matched_ts, joint_h5, tcp_h5 = extract_joint_and_tcp(
    H5_PATH, CAMERA_TIMESTAMP, ARM_SUFFIX, EXACT_H5_TIMESTAMP
)

if JOINT_SOURCE_MODE == 'manual_deg':
    joint_used = manual_joint_from_deg(MANUAL_JOINT_DEG, MANUAL_GRIPPER_WIDTH)
else:
    joint_used = joint_h5.copy()

active_urdf = Path(URDF_CANDIDATES[ACTIVE_URDF_KEY])
color_path, depth_path, color_img, depth_img = load_scene_images(SCENE_CAM_DIR, CAMERA_TIMESTAMP)
point_cloud = depth_to_point_cloud(
    depth_img,
    intrinsic,
    stride=POINT_STRIDE,
    depth_scale=DEPTH_SCALE,
    min_depth_m=MIN_DEPTH_M,
    max_depth_m=MAX_DEPTH_M,
    point_max=POINT_MAX,
    apply_o3d=APPLY_O3D_TO_POINT_CLOUD,
)

print('camera_timestamp       =', CAMERA_TIMESTAMP)
print('matched_h5_index       =', idx)
print('matched_h5_timestamp   =', matched_ts)
print('timestamp_diff_ms      =', abs(matched_ts - CAMERA_TIMESTAMP))
print('cam_base_mode          =', CAM_BASE_MODE)
print('active_urdf_key        =', ACTIVE_URDF_KEY)
print('active_urdf            =', active_urdf)
print('apply_extra_base_rpy   =', APPLY_EXTRA_BASE_RPY_ZYX_DEG)
print('extra_base_rpy_zyx_deg =', EXTRA_BASE_RPY_ZYX_DEG)
print('point_cloud shape      =', point_cloud.shape)
print('joint_source_mode      =', JOINT_SOURCE_MODE)
print('joint_h5               =', joint_h5)
print('joint_used             =', joint_used)
print('tcp_h5                 =', tcp_h5)
print('json pose parent       =', raw_json.get('parent_link_name'))
print('json cam_serial        =', raw_json.get('cam_serial'))
print('T_base_to_cam ='); print(T_base_to_cam)
print('cam_to_base(eval style) ='); print(cam_to_base)
if extra_base_T is not None:
    print('extra_base_T ='); print(extra_base_T)

display(Image.open(color_path))


In [ ]:
cur_transforms, visuals_map = robot_helper.forward_kinematic_single(
    joint=joint_used.astype(np.float32),
    joint_cfgs=JOINT_CFGS,
    is_rad=True,
    urdf_file=str(active_urdf),
    with_visuals_map=True,
)

fk_candidates = fk_tcp_candidates(joint_used, JOINT_CFGS, str(active_urdf))
print('link_count         =', len(cur_transforms))
print('fk candidate keys  =', list(fk_candidates.keys()))
for name, T in fk_candidates.items():
    print(f'{name} =\n', T)


In [ ]:
fig = go.Figure()

if point_cloud.shape[0] > 0:
    fig.add_trace(go.Scatter3d(
        x=point_cloud[:, 0],
        y=point_cloud[:, 1],
        z=point_cloud[:, 2],
        mode='markers',
        marker=dict(size=1.0, color=point_cloud[:, 2], colorscale='Viridis', opacity=0.35),
        name='depth_point_cloud',
    ))

mesh_color = 'rgba(80,160,255,0.62)'
for link, transform in cur_transforms.items():
    if not should_keep_link(link):
        continue
    for v in visuals_map[link]:
        if v.geom_param is None:
            continue
        verts, faces = load_mesh_vertices_faces(v.geom_param, str(active_urdf))
        faces = downsample_faces(faces, MESH_FACE_LIMIT)
        tf = compose_mesh_tf(cam_to_base, transform, v.offset, extra_base_T=extra_base_T)
        verts_tf = apply_transform(verts, tf)
        fig.add_trace(go.Mesh3d(
            x=verts_tf[:, 0],
            y=verts_tf[:, 1],
            z=verts_tf[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=mesh_color,
            opacity=0.58,
            name=f'mesh::{link}',
            showscale=False,
            hoverinfo='name',
        ))

if SHOW_FRAME_AXES:
    add_frame(fig, np.eye(4), 'camera', axis_len=FRAME_AXIS_LEN)
    base_link_tf = safe_link_tf(cur_transforms, 'base_link')
    if base_link_tf is not None:
        base_display = compose_fk_world(cam_to_base, base_link_tf, extra_base_T)
        base_display = remap_base_axes(base_display, BASE_AXIS_REMAP_MODE)
        add_frame(fig, base_display, f'display_base_{BASE_AXIS_REMAP_MODE}', axis_len=FRAME_AXIS_LEN)
    if SHOW_TCP_FRAME:
        add_frame(fig, compose_real_tcp_world(T_base_to_cam, tcp_h5), 'h5_tcp', axis_len=FRAME_AXIS_LEN * 0.8)
        if 'tcp_from_flange' in fk_candidates:
            add_frame(
                fig,
                compose_fk_world(cam_to_base, fk_candidates['tcp_from_flange'], extra_base_T),
                'fk_tcp',
                axis_len=FRAME_AXIS_LEN * 0.8,
            )

fig.update_layout(
    title=f'Single FOAR 3D Debug: mesh fixed, base axis remap={BASE_AXIS_REMAP_MODE}',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data',
    ),
    width=1300,
    height=950,
    showlegend=False,
)
display(fig)


In [ ]:
# 可选：快速扫一遍两套 URDF 的 base frame 差异。
for urdf_key, urdf_path in URDF_CANDIDATES.items():
    cur_transforms_probe, _ = robot_helper.forward_kinematic_single(
        joint=joint_used.astype(np.float32),
        joint_cfgs=JOINT_CFGS,
        is_rad=True,
        urdf_file=str(urdf_path),
        with_visuals_map=True,
    )
    print('===== ', urdf_key, ' =====')
    print('urdf_path =', urdf_path)
    print('link_count =', len(cur_transforms_probe))


In [ ]:

# 进一步排查：直接打印真实 JSON base 轴、URDF base_link 轴、h5 tcp 轴。
print('=== axis diagnostics ===')
print('json T_base_to_cam rotation columns (base axes in camera frame):')
print(T_base_to_cam[:3, :3])
print('json base_x =', T_base_to_cam[:3, 0])
print('json base_y =', T_base_to_cam[:3, 1])
print('json base_z =', T_base_to_cam[:3, 2])
print('NOTE: 这里如果 base_z 本来就是垂直桌面，那红轴目标不能靠显示 remap 硬修。')

base_link_tf = safe_link_tf(cur_transforms, 'base_link')
if base_link_tf is not None:
    base_display = compose_fk_world(cam_to_base, base_link_tf, extra_base_T)
    print('urdf/display base_link rotation columns:')
    print(base_display[:3, :3])
    print('urdf/display x =', base_display[:3, 0])
    print('urdf/display y =', base_display[:3, 1])
    print('urdf/display z =', base_display[:3, 2])

if 'tcp_from_flange' in fk_candidates:
    fk_tcp_world = compose_fk_world(cam_to_base, fk_candidates['tcp_from_flange'], extra_base_T)
    print('fk_tcp rotation columns:')
    print(fk_tcp_world[:3, :3])
    print('fk_tcp x =', fk_tcp_world[:3, 0])
    print('fk_tcp y =', fk_tcp_world[:3, 1])
    print('fk_tcp z =', fk_tcp_world[:3, 2])

print('h5 tcp rotation columns:')
print(compose_real_tcp_world(T_base_to_cam, tcp_h5)[:3, :3])
print('h5_tcp x =', compose_real_tcp_world(T_base_to_cam, tcp_h5)[:3, 0])
print('h5_tcp y =', compose_real_tcp_world(T_base_to_cam, tcp_h5)[:3, 1])
print('h5_tcp z =', compose_real_tcp_world(T_base_to_cam, tcp_h5)[:3, 2])
